# LoRA Fine-Tuning LLaMA-3B-Instruct
This notebook fine-tunes a HuggingFace LLaMA-3B-Instruct model using LoRA (PEFT).

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
# TRAIN_CSV = "task2/data/English_train.jsonl"
# VAL_CSV = "task2/data/clef2026_gpt4_o_mini_val.json"
# OUTPUT_DIR = "outputs"

In [49]:
!pip install -q transformers peft evaluate tomli

In [50]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split


import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate

from peft import LoraConfig, get_peft_model


In [51]:
torch.set_default_dtype(torch.float32)


In [52]:
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))


def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )


In [53]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=True,
        use_lora=False,
        is_base_encoder=True,
        lora_rank=8,
        lora_alpha=16,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)
        self.is_base_encoder = is_base_encoder

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj"],
                lora_dropout=0.0,
                bias="none",
            )
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        if self.is_base_encoder:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, -1, :]
        # if self.is_base_encoder:
        #     pooled_output = outputs.last_hidden_state[:, 0, :]
        # else:
        #     pooled_output = outputs.last_hidden_state[:, -1, :]
        pooled_output = pooled_output.to(torch.device("cuda"), dtype=torch.float)
        #pooled_output = pooled_output.float()
        #self.classifier = self.classifier.to(torch.device("cuda"),torch.float16)
        logits = self.classifier(pooled_output)
        return logits


In [54]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        # if self.tokenizer.pad_token is None:
        #     self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


In [55]:
class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        epochs,
        lr,
        patience,
        output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch+1}/{self.epochs}")
            self.model.train()

            total_loss, total_acc = 0, 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                logits = self.model(input_ids, attention_mask)

                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()
                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")
        tokenizer.save_pretrained(self.output_dir)
        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model_epoch_{epoch}"),
        )


In [56]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp,
        device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        # if self.tokenizer.pad_token is None:
        #     self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = CustomClassifier(base_model, use_lora=True,is_base_encoder=False,)
        #self.model = CustomClassifier(base_model, use_lora=False, is_base_encoder=True) # bert
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def encode_input(self, claim, questions, verdict, justification, max_length=150):

        text = f"Claim: {claim}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, questions, verdict, justification):
        ids, mask = self.encode_input(claim, questions, verdict, justification)
        with torch.no_grad():
            return float(self.model(ids, mask).item())


In [57]:
# ===== PATH PLACEHOLDERS =====
TRAIN_CSV = "/content/drive/MyDrive/AIR_CheckThat/data/english/english_train.json"
VAL_CSV = "/content/drive/MyDrive/AIR_CheckThat/data/english/clef2026_gpt4_o_mini_val.json"
# TEST_JSON = "/kaggle/input/YOUR_DATA/test.jsonl"

OUTPUT_DIR = "/content/drive/MyDrive/AIR_CheckThat/Training_reward_models"

BASE_MODEL = "meta-llama/Llama-3.2-1B"
#BASE_MODEL = "distilbert-base-uncased" # bert open source


MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 3
LR = 1e-4


In [70]:
%cd AIR_Group_Task
!git pull

/content/AIR_Group_Task
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.04 KiB | 1.04 MiB/s, done.
From https://github.com/ayadssk/AIR_Group_Task
   aa4b881..28d5cae  main       -> origin/main
Updating aa4b881..28d5cae
Fast-forward
 task2/reasoning_trace_build.py | 3 ++-
 1 file changed, 2 insertions(+), 1 deletion(-)


In [77]:
%cd /content

/content


In [79]:
!mkdir -p output/training_data_for_RM
!python AIR_Group_Task/task2/reasoning_trace_build.py \
  --input "/content/drive/MyDrive/AIR_CheckThat/data/english/english_train.json" \
  --output "output/training_data_for_RM/english_train.jsonl"

Wrote 31433 examples to output/training_data_for_RM/english_train.jsonl


In [80]:
TRAIN_CSV = "output/training_data_for_RM/english_train.jsonl"

In [81]:
from huggingface_hub import login
login()
train_df = pd.read_json(TRAIN_CSV, lines=True)


In [ ]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=42
)
# val_df = pd.read_json(VAL_CSV, lines = True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
)
# model = CustomClassifier(
#     BASE_MODEL,
#     use_lora=False,
#     is_base_encoder=True,
# )

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
)

trainer.train()


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

trainable params: 1181697 || all params: 1236996097 || trainable%: 0.10

Epoch 1/3


  0%|          | 11/6287 [00:23<3:47:18,  2.17s/it]

In [ ]:
import os

print(TRAIN_CSV)
print("exists:", os.path.exists(TRAIN_CSV))
print("size MB:", os.path.getsize(TRAIN_CSV) / 1024 / 1024)

with open(TRAIN_CSV, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f.readline()[:300])

In [ ]:
with open(VAL_CSV, "r") as f:
  test_data = json.load(f)

predictions = []

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/model_epoch_2",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
)

# Example scoring
for idx, sample in enumerate(test_data):
  verdict_list = []
  verifier_score_list = []
  justification_list = []
  approved_majority_list = []
  for decoding_sample in range(1, len(sample["Reasoning_traces"]) + 1):
    justification = (
        remove_label_pattern(
            sample["Reasoning_traces"][decoding_sample - 1]
        ).split("Label:")[0]
    )
    score = evaluator.score(
    sample["claim"],
    sample.get("Questions", ""),
    sample["Verdict_list"][decoding_sample - 1].lower(),
    justification,
)
    verdict_list.append(sample["Verdict_list"][decoding_sample - 1])
    justification_list.append(justification)
    verifier_score_list.append(score)
    best_verdict = verdict_list[np.argmax(np.array(verifier_score_list))]
    predictions.append({
        "query_id": idx,
        "Claim": sample["claim"],
        "Label": sample["label"],
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })




In [ ]:
# with open(f"/content/drive/MyDrive/AIR_CheckThat/Training_reward_models/clef_predictions.json", "w") as fp:
#   json.dump(predictions, fp, indent=4)

In [ ]:
os.makedirs("output/RM_prediction", exist_ok=True)

with open("output/RM_prediction/clef_predictions.json", "w", encoding="utf-8") as fp:
    json.dump(predictions, fp, indent=4)

In [ ]:
!python task2/scorer.py